In [4]:
from pathlib import Path
import gc
import json
import time
import warnings

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

warnings.filterwarnings("ignore")

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "packages/network-security-ai/pyproject.toml").is_file())

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_PATH = (
    PROJECT_ROOT
    / "models/network/lightgbm_top40_v3.txt"
)

FEATURE_PATH = (
    PROJECT_ROOT
    / "results/feature_selection/top40_features.csv"
)

LABEL_PATH = (
    PROJECT_ROOT
    / "data/processed/network_ml/label_mapping.json"
)

TEST_PATH = (
    PROJECT_ROOT
    / "data/processed/network_ml_split_v2/test.parquet"
)

RESULT_DIR = (
    PROJECT_ROOT
    / "results/final_evaluation"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("MODEL        :", MODEL_PATH)
print("FEATURE      :", FEATURE_PATH)
print("LABEL        :", LABEL_PATH)
print("TEST :", TEST_PATH)
print("RESULT       :", RESULT_DIR)

PROJECT_ROOT : /home/geonug/kdt-linux/ML
MODEL        : /home/geonug/kdt-linux/ML/models/network/lightgbm_top40_v3.txt
FEATURE      : /home/geonug/kdt-linux/ML/results/feature_selection/top40_features.csv
LABEL        : /home/geonug/kdt-linux/ML/data/processed/network_ml/label_mapping.json
TEST : /home/geonug/kdt-linux/ML/data/processed/network_ml_split_v2/test.parquet
RESULT       : /home/geonug/kdt-linux/ML/results/final_evaluation


In [5]:
print("Model exists   :", MODEL_PATH.exists())
print("Feature exists :", FEATURE_PATH.exists())
print("Label exists   :", LABEL_PATH.exists())
print("Test exists    :", TEST_PATH.exists())

assert MODEL_PATH.exists(), MODEL_PATH
assert FEATURE_PATH.exists(), FEATURE_PATH
assert LABEL_PATH.exists(), LABEL_PATH
assert TEST_PATH.exists(), TEST_PATH

print(
    f"\nTest file size: "
    f"{TEST_PATH.stat().st_size / 1024**2:.1f} MB"
)

print("\n환경 확인 완료")

Model exists   : True
Feature exists : True
Label exists   : True
Test exists    : True

Test file size: 232.7 MB

환경 확인 완료


In [6]:
feature_df = pd.read_csv(FEATURE_PATH)

print(feature_df.head())
print()
print("columns:", feature_df.columns.tolist())

   rank            feature
0     1  Fwd Act Data Pkts
1     2   Fwd Seg Size Min
2     3     Fwd Header Len
3     4       ECE Flag Cnt
4     5  Init Fwd Win Byts

columns: ['rank', 'feature']


In [7]:
possible_columns = [
    "feature",
    "Feature",
    "feature_name",
    "Feature Name",
]

feature_col = None

for col in possible_columns:
    if col in feature_df.columns:
        feature_col = col
        break

if feature_col is None:
    object_cols = (
        feature_df
        .select_dtypes(include="object")
        .columns
        .tolist()
    )

    if not object_cols:
        raise ValueError(
            f"Feature column not found: "
            f"{feature_df.columns.tolist()}"
        )

    feature_col = object_cols[0]

TOP40_FEATURES = (
    feature_df[feature_col]
    .astype(str)
    .tolist()
)

assert len(TOP40_FEATURES) == 40

print("Feature count:", len(TOP40_FEATURES))

for i, feature in enumerate(
    TOP40_FEATURES,
    1,
):
    print(f"{i:02d}. {feature}")

Feature count: 40
01. Fwd Act Data Pkts
02. Fwd Seg Size Min
03. Fwd Header Len
04. ECE Flag Cnt
05. Init Fwd Win Byts
06. URG Flag Cnt
07. RST Flag Cnt
08. Dst Port
09. Fwd PSH Flags
10. Fwd Pkts/s
11. Fwd Pkt Len Mean
12. Pkt Len Max
13. SYN Flag Cnt
14. Fwd Seg Size Avg
15. Bwd Header Len
16. Init Bwd Win Byts
17. Fwd IAT Std
18. Fwd IAT Max
19. Pkt Size Avg
20. Fwd IAT Tot
21. Bwd Pkt Len Max
22. PSH Flag Cnt
23. Flow IAT Mean
24. Bwd IAT Min
25. Bwd Pkt Len Std
26. Pkt Len Mean
27. Bwd IAT Max
28. Fwd Pkt Len Std
29. ACK Flag Cnt
30. TotLen Bwd Pkts
31. Pkt Len Std
32. FIN Flag Cnt
33. Bwd IAT Mean
34. Flow Duration
35. Flow IAT Max
36. Subflow Bwd Byts
37. Idle Max
38. Fwd URG Flags
39. Bwd Pkts/s
40. Bwd IAT Std


In [8]:
with open(
    LABEL_PATH,
    "r",
    encoding="utf-8",
) as f:
    raw_label_mapping = json.load(f)

print(raw_label_mapping)

{'Benign': 0, 'Bot': 1, 'Brute Force -Web': 2, 'Brute Force -XSS': 3, 'DDOS attack-HOIC': 4, 'DDOS attack-LOIC-UDP': 5, 'DDoS attacks-LOIC-HTTP': 6, 'DoS attacks-GoldenEye': 7, 'DoS attacks-Hulk': 8, 'DoS attacks-SlowHTTPTest': 9, 'DoS attacks-Slowloris': 10, 'FTP-BruteForce': 11, 'Infilteration': 12, 'SQL Injection': 13, 'SSH-Bruteforce': 14}


In [9]:
CLASS_NAMES = [
    "Benign",
    "Bot",
    "Brute Force -Web",
    "Brute Force -XSS",
    "DDOS attack-HOIC",
    "DDOS attack-LOIC-UDP",
    "DDoS attacks-LOIC-HTTP",
    "DoS attacks-GoldenEye",
    "DoS attacks-Hulk",
    "DoS attacks-SlowHTTPTest",
    "DoS attacks-Slowloris",
    "FTP-BruteForce",
    "Infilteration",
    "SQL Injection",
    "SSH-Bruteforce",
]

N_CLASSES = len(CLASS_NAMES)

print("Number of classes:", N_CLASSES)

for i, name in enumerate(CLASS_NAMES):
    print(i, name)

Number of classes: 15
0 Benign
1 Bot
2 Brute Force -Web
3 Brute Force -XSS
4 DDOS attack-HOIC
5 DDOS attack-LOIC-UDP
6 DDoS attacks-LOIC-HTTP
7 DoS attacks-GoldenEye
8 DoS attacks-Hulk
9 DoS attacks-SlowHTTPTest
10 DoS attacks-Slowloris
11 FTP-BruteForce
12 Infilteration
13 SQL Injection
14 SSH-Bruteforce


In [10]:
model = lgb.Booster(
    model_file=str(MODEL_PATH)
)

print("Model loaded")
print("Features:", model.num_feature())
print("Classes :", model.num_model_per_iteration())

assert model.num_feature() == 40

Model loaded
Features: 40
Classes : 15


In [12]:
model_features = model.feature_name()

print("Model feature count :", len(model_features))
print("CSV feature count   :", len(TOP40_FEATURES))

assert len(model_features) == len(TOP40_FEATURES) == 40


# LightGBM이 NumPy 입력으로 학습되었을 때 생성하는
# Column_0 ~ Column_39 형태인지 확인
expected_generic_names = [
    f"Column_{i}"
    for i in range(len(TOP40_FEATURES))
]

uses_generic_feature_names = (
    model_features == expected_generic_names
)

print(
    "Generic LightGBM feature names :",
    uses_generic_feature_names,
)


if uses_generic_feature_names:

    print(
        "\n[INFO]"
        "\n모델이 NumPy 배열 기반으로 학습되어"
        "\n실제 Feature 이름 대신 Column_0 ~ Column_39가 저장되었습니다."
    )

    print(
        "\n따라서 모델 자체에서는 Feature 이름을 이용한"
        "\n순서 검증이 불가능합니다."
    )

    print(
        "\n평가 입력은 학습에 사용한 Top40 CSV 순서를"
        "\n그대로 사용합니다."
    )

else:

    feature_order_match = (
        model_features == TOP40_FEATURES
    )

    print(
        "Feature order match :",
        feature_order_match,
    )

    if not feature_order_match:

        for i, (model_feature, csv_feature) in enumerate(
            zip(
                model_features,
                TOP40_FEATURES,
            )
        ):
            if model_feature != csv_feature:
                print(
                    f"Mismatch [{i}] "
                    f"model={model_feature!r} "
                    f"csv={csv_feature!r}"
                )

        raise ValueError(
            "실제 Feature 이름이 저장된 모델과 "
            "Top40 CSV의 Feature 순서가 다릅니다."
        )


print("\n=== Evaluation Feature Order ===")

for i, feature in enumerate(
    TOP40_FEATURES
):
    print(
        f"Column_{i:<2} -> {feature}"
    )

Model feature count : 40
CSV feature count   : 40
Generic LightGBM feature names : True

[INFO]
모델이 NumPy 배열 기반으로 학습되어
실제 Feature 이름 대신 Column_0 ~ Column_39가 저장되었습니다.

따라서 모델 자체에서는 Feature 이름을 이용한
순서 검증이 불가능합니다.

평가 입력은 학습에 사용한 Top40 CSV 순서를
그대로 사용합니다.

=== Evaluation Feature Order ===
Column_0  -> Fwd Act Data Pkts
Column_1  -> Fwd Seg Size Min
Column_2  -> Fwd Header Len
Column_3  -> ECE Flag Cnt
Column_4  -> Init Fwd Win Byts
Column_5  -> URG Flag Cnt
Column_6  -> RST Flag Cnt
Column_7  -> Dst Port
Column_8  -> Fwd PSH Flags
Column_9  -> Fwd Pkts/s
Column_10 -> Fwd Pkt Len Mean
Column_11 -> Pkt Len Max
Column_12 -> SYN Flag Cnt
Column_13 -> Fwd Seg Size Avg
Column_14 -> Bwd Header Len
Column_15 -> Init Bwd Win Byts
Column_16 -> Fwd IAT Std
Column_17 -> Fwd IAT Max
Column_18 -> Pkt Size Avg
Column_19 -> Fwd IAT Tot
Column_20 -> Bwd Pkt Len Max
Column_21 -> PSH Flag Cnt
Column_22 -> Flow IAT Mean
Column_23 -> Bwd IAT Min
Column_24 -> Bwd Pkt Len Std
Column_25 -> Pkt Len Mean
Column_26

In [13]:
VALID_PATH = (
    PROJECT_ROOT
    / "data/processed/network_ml_dev_v3/model_sample"
    / "valid_v3_sample.parquet"
)

assert VALID_PATH.exists(), VALID_PATH

print(
    "Validation:",
    VALID_PATH
)

Validation: /home/geonug/kdt-linux/ML/data/processed/network_ml_dev_v3/model_sample/valid_v3_sample.parquet


In [14]:
valid_df = pd.read_parquet(
    VALID_PATH,
    columns=TOP40_FEATURES + ["label_id"],
)

X_valid_check = (
    valid_df[TOP40_FEATURES]
    .to_numpy(
        dtype=np.float32,
        copy=True,
    )
)

y_valid_check = (
    valid_df["label_id"]
    .to_numpy(
        dtype=np.int16,
        copy=True,
    )
)

del valid_df
gc.collect()

X_valid_check[
    ~np.isfinite(X_valid_check)
] = np.nan

print(
    "Validation shape:",
    X_valid_check.shape,
)

Validation shape: (142082, 40)


In [15]:
valid_probability_check = model.predict(
    X_valid_check
)

valid_prediction_check = np.argmax(
    valid_probability_check,
    axis=1,
).astype(np.int16)

In [16]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
)

check_accuracy = accuracy_score(
    y_valid_check,
    valid_prediction_check,
)

(
    check_macro_precision,
    check_macro_recall,
    check_macro_f1,
    _
) = precision_recall_fscore_support(
    y_valid_check,
    valid_prediction_check,
    average="macro",
    zero_division=0,
)

_, _, check_weighted_f1, _ = (
    precision_recall_fscore_support(
        y_valid_check,
        valid_prediction_check,
        average="weighted",
        zero_division=0,
    )
)

print("=== Validation Reproduction ===")
print(
    f"Accuracy        : {check_accuracy:.6f}"
)
print(
    f"Macro Precision : {check_macro_precision:.6f}"
)
print(
    f"Macro Recall    : {check_macro_recall:.6f}"
)
print(
    f"Macro F1        : {check_macro_f1:.6f}"
)
print(
    f"Weighted F1     : {check_weighted_f1:.6f}"
)

=== Validation Reproduction ===
Accuracy        : 0.945792
Macro Precision : 0.871111
Macro Recall    : 0.885821
Macro F1        : 0.876563
Weighted F1     : 0.945745


In [17]:
del X_valid_check
del y_valid_check
del valid_probability_check
del valid_prediction_check

gc.collect()

print("Validation arrays released.")

Validation arrays released.


In [18]:
import pyarrow.parquet as pq

parquet_file = pq.ParquetFile(TEST_PATH)

print("=== Test V2 Metadata ===")
print(
    "Rows       :",
    f"{parquet_file.metadata.num_rows:,}"
)
print(
    "Row groups :",
    parquet_file.num_row_groups
)
print(
    "Columns    :",
    parquet_file.metadata.num_columns
)
print(
    "File size  :",
    f"{TEST_PATH.stat().st_size / 1024**2:.1f} MB"
)

=== Test V2 Metadata ===
Rows       : 2,426,804
Row groups : 20
Columns    : 84
File size  : 232.7 MB


In [19]:
total_confusion = np.zeros(
    (N_CLASSES, N_CLASSES),
    dtype=np.int64,
)

total_samples = 0
total_correct = 0
total_predict_seconds = 0.0

batch_stats = []

print("Accumulator initialized.")

Accumulator initialized.


In [20]:
BATCH_SIZE = 100_000

LOAD_COLUMNS = TOP40_FEATURES + ["label_id"]

parquet_file = pq.ParquetFile(TEST_PATH)

evaluation_start = time.perf_counter()

batch_iterator = parquet_file.iter_batches(
    batch_size=BATCH_SIZE,
    columns=LOAD_COLUMNS,
)

for batch_idx, batch in enumerate(
    batch_iterator,
    start=1,
):
    batch_start = time.perf_counter()

    # 현재 batch만 메모리에 적재
    df = batch.to_pandas()

    X_chunk = df[TOP40_FEATURES].to_numpy(
        dtype=np.float32,
        copy=True,
    )

    y_chunk = df["label_id"].to_numpy(
        dtype=np.int16,
        copy=True,
    )

    del df
    del batch

    # 기존 학습/검증과 동일하게
    # inf를 NaN으로 처리
    X_chunk[
        ~np.isfinite(X_chunk)
    ] = np.nan

    # -------------------------
    # Prediction
    # -------------------------
    predict_start = time.perf_counter()

    probabilities = model.predict(
        X_chunk
    )

    predict_seconds = (
        time.perf_counter()
        - predict_start
    )

    predictions = np.argmax(
        probabilities,
        axis=1,
    ).astype(np.int16)

    # -------------------------
    # Confusion Matrix
    # -------------------------
    cm = confusion_matrix(
        y_chunk,
        predictions,
        labels=np.arange(N_CLASSES),
    )

    total_confusion += cm

    n_samples = len(y_chunk)

    correct = int(
        (predictions == y_chunk).sum()
    )

    total_samples += n_samples
    total_correct += correct
    total_predict_seconds += predict_seconds

    batch_seconds = (
        time.perf_counter()
        - batch_start
    )

    batch_stats.append(
        {
            "batch": batch_idx,
            "samples": n_samples,
            "correct": correct,
            "accuracy": correct / n_samples,
            "predict_seconds": predict_seconds,
            "total_seconds": batch_seconds,
        }
    )

    print(
        f"[Batch {batch_idx:03d}] "
        f"n={n_samples:>8,} "
        f"acc={correct/n_samples:.4f} "
        f"predict={predict_seconds:.2f}s "
        f"total={batch_seconds:.2f}s"
    )

    # 현재 batch 즉시 해제
    del X_chunk
    del y_chunk
    del probabilities
    del predictions
    del cm

    gc.collect()


evaluation_seconds = (
    time.perf_counter()
    - evaluation_start
)

print("\n=== TEST COMPLETE ===")
print(
    f"Samples : {total_samples:,}"
)
print(
    f"Correct : {total_correct:,}"
)
print(
    f"Accuracy: "
    f"{total_correct / total_samples:.6f}"
)
print(
    f"Predict : "
    f"{total_predict_seconds:.2f}s"
)
print(
    f"Total   : "
    f"{evaluation_seconds:.2f}s"
)

[Batch 001] n= 100,000 acc=0.8238 predict=31.89s total=32.03s
[Batch 002] n= 100,000 acc=0.5231 predict=34.46s total=34.74s
[Batch 003] n= 100,000 acc=1.0000 predict=36.25s total=36.63s
[Batch 004] n= 100,000 acc=0.7014 predict=34.24s total=34.88s
[Batch 005] n= 100,000 acc=0.7755 predict=33.07s total=33.16s
[Batch 006] n= 100,000 acc=0.9662 predict=33.90s total=33.94s
[Batch 007] n= 100,000 acc=0.7346 predict=32.66s total=32.79s
[Batch 008] n= 100,000 acc=0.7152 predict=33.25s total=33.30s
[Batch 009] n= 100,000 acc=0.5020 predict=34.07s total=34.17s
[Batch 010] n= 100,000 acc=0.4760 predict=33.62s total=34.43s
[Batch 011] n= 100,000 acc=0.4633 predict=30.83s total=31.04s
[Batch 012] n= 100,000 acc=0.4524 predict=32.69s total=32.77s
[Batch 013] n= 100,000 acc=0.4571 predict=32.07s total=32.10s
[Batch 014] n= 100,000 acc=0.4668 predict=32.66s total=32.73s
[Batch 015] n= 100,000 acc=0.7355 predict=33.22s total=33.33s
[Batch 016] n= 100,000 acc=0.7740 predict=32.77s total=33.20s
[Batch 0

In [21]:
batch_stats_df = pd.DataFrame(
    batch_stats
)

batch_stats_df.to_csv(
    RESULT_DIR / "test_batch_stats.csv",
    index=False,
)

confusion_df = pd.DataFrame(
    total_confusion,
    index=CLASS_NAMES,
    columns=CLASS_NAMES,
)

confusion_df.to_csv(
    RESULT_DIR / "test_confusion_matrix.csv"
)

np.save(
    RESULT_DIR / "test_confusion_matrix.npy",
    total_confusion,
)

print("Final Test checkpoint saved.")

print(
    RESULT_DIR
    / "test_confusion_matrix.csv"
)

Final Test checkpoint saved.
/home/geonug/kdt-linux/ML/results/final_evaluation/test_confusion_matrix.csv


In [22]:
cm = total_confusion.astype(
    np.float64
)

tp = np.diag(cm)
support = cm.sum(axis=1)
predicted = cm.sum(axis=0)

fp = predicted - tp
fn = support - tp

precision = np.divide(
    tp,
    tp + fp,
    out=np.zeros_like(tp),
    where=(tp + fp) != 0,
)

recall = np.divide(
    tp,
    tp + fn,
    out=np.zeros_like(tp),
    where=(tp + fn) != 0,
)

f1 = np.divide(
    2 * precision * recall,
    precision + recall,
    out=np.zeros_like(tp),
    where=(precision + recall) != 0,
)

accuracy = (
    tp.sum()
    / cm.sum()
)

macro_precision = precision.mean()
macro_recall = recall.mean()
macro_f1 = f1.mean()

weighted_f1 = np.average(
    f1,
    weights=support,
)

print("=== FINAL TEST METRICS ===")
print(
    f"Accuracy        : {accuracy:.6f}"
)
print(
    f"Macro Precision : {macro_precision:.6f}"
)
print(
    f"Macro Recall    : {macro_recall:.6f}"
)
print(
    f"Macro F1        : {macro_f1:.6f}"
)
print(
    f"Weighted F1     : {weighted_f1:.6f}"
)

=== FINAL TEST METRICS ===
Accuracy        : 0.689660
Macro Precision : 0.647634
Macro Recall    : 0.787883
Macro F1        : 0.656252
Weighted F1     : 0.795021


In [23]:
per_class_df = pd.DataFrame({
    "class_id": np.arange(N_CLASSES),
    "class": CLASS_NAMES,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "support": support.astype(np.int64),
})

print(
    per_class_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

 class_id                    class  precision   recall       f1  support
        0                   Benign   0.997435 0.653062 0.789322  2020044
        1                      Bot   0.999021 0.999958 0.999489    47967
        2         Brute Force -Web   0.092398 0.929412 0.168085       85
        3         Brute Force -XSS   0.695652 0.888889 0.780488       36
        4         DDOS attack-HOIC   1.000000 1.000000 1.000000   105263
        5     DDOS attack-LOIC-UDP   0.778125 0.961390 0.860104      259
        6   DDoS attacks-LOIC-HTTP   0.999677 0.999146 0.999411    86646
        7    DoS attacks-GoldenEye   0.998387 0.999838 0.999112     6190
        8         DoS attacks-Hulk   0.999971 1.000000 0.999985    68155
        9 DoS attacks-SlowHTTPTest   0.011655 0.000315 0.000614    47550
       10    DoS attacks-Slowloris   0.995065 1.000000 0.997526     1613
       11           FTP-BruteForce   0.000000 0.000000 0.000000     1272
       12            Infilteration   0.028875 0.859

In [24]:
error_rows = []

for true_id in range(N_CLASSES):
    for pred_id in range(N_CLASSES):

        if true_id == pred_id:
            continue

        count = int(
            total_confusion[
                true_id,
                pred_id,
            ]
        )

        if count == 0:
            continue

        error_rows.append({
            "true_id": true_id,
            "true_class": CLASS_NAMES[true_id],
            "pred_id": pred_id,
            "pred_class": CLASS_NAMES[pred_id],
            "count": count,
        })


error_df = (
    pd.DataFrame(error_rows)
    .sort_values(
        "count",
        ascending=False,
    )
)

print(
    error_df.head(30).to_string(
        index=False
    )
)

 true_id               true_class  pred_id               pred_class  count
       0                   Benign       12            Infilteration 699881
       9 DoS attacks-SlowHTTPTest       11           FTP-BruteForce  47535
      12            Infilteration        0                   Benign   3388
      11           FTP-BruteForce        9 DoS attacks-SlowHTTPTest   1272
       0                   Benign        2         Brute Force -Web    770
       0                   Benign       13            SQL Injection     73
       6   DDoS attacks-LOIC-HTTP        5     DDOS attack-LOIC-UDP     71
       0                   Benign        1                      Bot     47
       0                   Benign        6   DDoS attacks-LOIC-HTTP     17
       0                   Benign       14           SSH-Bruteforce     13
       5     DDOS attack-LOIC-UDP        6   DDoS attacks-LOIC-HTTP     10
       0                   Benign        7    DoS attacks-GoldenEye     10
       0                 

In [25]:
test_distribution = pd.DataFrame({
    "class_id": np.arange(N_CLASSES),
    "class": CLASS_NAMES,
    "support": support.astype(np.int64),
})

test_distribution["ratio"] = (
    test_distribution["support"]
    / test_distribution["support"].sum()
)

print(
    test_distribution.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

 class_id                    class  support    ratio
        0                   Benign  2020044 0.832389
        1                      Bot    47967 0.019766
        2         Brute Force -Web       85 0.000035
        3         Brute Force -XSS       36 0.000015
        4         DDOS attack-HOIC   105263 0.043375
        5     DDOS attack-LOIC-UDP      259 0.000107
        6   DDoS attacks-LOIC-HTTP    86646 0.035704
        7    DoS attacks-GoldenEye     6190 0.002551
        8         DoS attacks-Hulk    68155 0.028084
        9 DoS attacks-SlowHTTPTest    47550 0.019594
       10    DoS attacks-Slowloris     1613 0.000665
       11           FTP-BruteForce     1272 0.000524
       12            Infilteration    24200 0.009972
       13            SQL Injection       19 0.000008
       14           SSH-Bruteforce    17505 0.007213
